In [ ]:
!pip install transformers peft torch accelerate google-genai

In [ ]:
import torch
import numpy as np

from transformers import DistilBertTokenizerFast
from transformers import DistilBertForSequenceClassification

from peft import PeftModel

from google import genai

In [ ]:
!unzip tier_c_model.zip

Archive:  tier_c_model.zip
  inflating: tokenizer_config.json   
  inflating: adapter_model.safetensors  
  inflating: README.md               
  inflating: training_args.bin       
  inflating: adapter_config.json     
  inflating: tokenizer.json          


In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 22.7 MB/s eta 0:00:00


In [ ]:
from transformers import DistilBertForSequenceClassification
from peft import PeftModel

base_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model = PeftModel.from_pretrained(base_model, ".")
model.eval()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSelfAttention(
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  )
      

In [ ]:
from google import genai
client = genai.Client(
    api_key="my_api_key"
)

In [ ]:
import time

population = []

topic = "Scientific Discovery and Technology"

for i in range(10):

    prompt = f"""
Write one paragraph (100-150 words)
about {topic}.

Do NOT imitate any author.
Only write the paragraph.
"""

    success = False

    while not success:
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=prompt
            )

            population.append(response.text)
            success = True

        except Exception as e:
            print("Retrying in 10 seconds...")
            time.sleep(10)

print(len(population))

Retrying in 10 seconds...
Retrying in 10 seconds...
Retrying in 10 seconds...
Retrying in 10 seconds...
Retrying in 10 seconds...
10


In [ ]:
import torch
import torch.nn.functional as F

def human_probability(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)

    # index 0 = Human
    return probs[0][0].item()

In [ ]:
print("population" in globals())
print("model" in globals())
print("tokenizer" in globals())
print("human_probability" in globals())

False
True
False
True


In [ ]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(".")

In [ ]:
print("tokenizer" in globals())

True


In [ ]:
with open("population.txt", "r", encoding="utf-8") as f:
    for i in range(20):
        print(i+1, repr(f.readline()))

1 'Paragraph 1 – Synthetic Biology\n'
2 '\n'
3 'Synthetic biology has moved from the theoretical fringe to a cornerstone of modern industrial manufacturing. By treating genetic code as a programmable language, scientists can engineer microbes to produce complex molecules, ranging from biofuels to pharmaceutical precursors. This shift represents a departure from traditional chemical synthesis, which often relies on fossil fuels and harsh catalysts. Instead, bioreactors filled with modified yeast or bacteria serve as microscopic factories that operate at room temperature. The precision of these biological systems allows for the creation of materials with specific structural properties that are difficult to achieve through conventional means. As the cost of DNA synthesis continues to fall, the potential for customized, bio-based solutions to global supply chain issues grows exponentially. However, the release of engineered organisms into the environment remains a concern, necessitating ri

In [ ]:
import re

with open("population.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Split on "Paragraph X - Title"
parts = re.split(r'Paragraph\s+\d+\s+[–-]\s+.*?\n\n', text)

population = [p.strip() for p in parts if p.strip()]

print("Paragraphs:", len(population))

Paragraphs: 10


In [ ]:
print("human_probability" in globals())

True


In [ ]:
print("model:", "model" in globals())
print("tokenizer:", "tokenizer" in globals())

model: True
tokenizer: True


In [ ]:
scores = []

for paragraph in population:
    score = human_probability(paragraph)
    scores.append((score, paragraph))

scores.sort(reverse=True)

top3 = scores[:3]

for i, (score, paragraph) in enumerate(top3, 1):
    print(f"\nWinner {i}")
    print(f"Human Probability: {score:.4f}")
    print(paragraph[:300])

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!



Winner 1
Human Probability: 0.1906
The development of mRNA vaccine technology represents a paradigm shift in how we approach infectious diseases and immunology. Unlike traditional vaccines, which often use weakened or inactivated viruses to trigger an immune response, mRNA platforms provide the body with the genetic instructions to b

Winner 2
Human Probability: 0.1892
Graphene, a single layer of carbon atoms arranged in a hexagonal lattice, continues to fascinate researchers with its extraordinary mechanical and electrical properties. Since its isolation, this two-dimensional material has promised to revolutionize everything from electronics to structural enginee

Winner 3
Human Probability: 0.1706
The transition toward 6G telecommunications is expected to usher in the "Internet of Senses," where digital experiences become indistinguishable from reality. Unlike its predecessors, 6G will utilize sub-terahertz frequencies to provide data transfer speeds that are orders of magnitude fas

In [ ]:
winners = [paragraph for score, paragraph in top3]

for i, w in enumerate(winners, 1):
    print("="*80)
    print(f"Winner {i}")
    print(w)

Winner 1
The development of mRNA vaccine technology represents a paradigm shift in how we approach infectious diseases and immunology. Unlike traditional vaccines, which often use weakened or inactivated viruses to trigger an immune response, mRNA platforms provide the body with the genetic instructions to build a specific viral protein. This approach allows for incredibly rapid development and manufacturing, as the template can be easily updated to target new variants or entirely different pathogens. The success of this technology during recent global health crises has opened the door for its use in treating cancer, where personalized vaccines could train the immune system to recognize and destroy tumor cells. Researchers are also exploring the use of mRNA to treat rare genetic disorders by replacing missing or defective proteins within the body. While the delivery mechanism—often involving lipid nanoparticles—requires careful engineering to ensure stability, the versatility of the pl

In [ ]:
mutated1 = """
The advent of mRNA platforms marks a fundamental departure from established immunological practices. Historical immunizations typically relied on attenuated pathogens to goad the immune system, whereas these novel blueprints instruct the body's own machinery to forge specific viral antigens. Rapidity is the hallmark of this system. Because the underlying code is digital, it can be adjusted with relative ease to combat emerging variants or entirely disparate diseases. This technological puissance was validated during recent pandemics, sparking interest in oncological applications where customized shots might teach our cells to annihilate malignancies. Furthermore, the potential to rectify hereditary flaws by substituting malformed proteins is being rigorously examined. A group of researchers were surprised by how effectively lipid envelopes stabilize the payload during delivery. By converting the human physique into a bespoke biological foundry, we have entered the age of targeted healing.
"""

In [ ]:
mutated2 = """
Arranged in a delicate hexagonal lattice, graphene consists of a solitary layer of carbon atoms that continues to startle the scientific community. This two-dimensional substance has, since its initial isolation, beckoned a new epoch for structural engineering and microelectronics alike. It is incredibly gossamer. Nevertheless, its internal resilience far outstrips that of industrial steel, providing a ruggedness that belies its weight. Because it permits the flow of thermal and electrical energy with negligible friction, it stands as the premier candidate for future processing units. Maugre these staggering capabilities, the transition to high-volume manufacturing at a sustainable price point remains elusive. Laboratory triumphs have yet to fully translate to the factory floor, though researchers are refining chemical vapor techniques to bridge this divide. Neither the cost nor the technical complexity provide an easy path toward mass adoption. The metamorphosis of a common mineral into a revolutionary tool underscores how microscopic insights reshape our macro world.
"""

mutated3 = """
Societal expectations for 6G telecommunications center on the arrival of a sensory internet, a realm where the boundary between tangible and simulated environments grows increasingly thin. By tapping into sub-terahertz bands, these networks will facilitate data transmission speeds far exceeding contemporary benchmarks. This immense capacity allows for the fluid incorporation of haptic responses. Consequently, a person might interact with remote objects and perceive their temperature or grain as if they were physically present. We may see a surgeon operate on a patient located across an ocean ere the decade is out, guided by instantaneous tactile feedback. Beyond mere communication, the infrastructure itself acts as a pervasive radar, utilizing high-frequency waves to chart surroundings with surgical precision. The range of available frequencies allow for centimeter-level mapping of dense urban environments. As these hurdles are cleared, the division between our carbon-based existence and the digital ether will likely dissolve.
"""

In [ ]:
scores = []

for paragraph in mutated_population:
    score = human_probability(paragraph)
    scores.append((score, paragraph))

scores.sort(reverse=True)

for score, paragraph in scores:
    print("Human Probability:", score)
    print("-"*80)
    print(paragraph[:300])

Human Probability: 0.19429481029510498
--------------------------------------------------------------------------------

Arranged in a delicate hexagonal lattice, graphene consists of a solitary layer of carbon atoms that continues to startle the scientific community. This two-dimensional substance has, since its initial isolation, beckoned a new epoch for structural engineering and microelectronics alike. It is incr
Human Probability: 0.1840442717075348
--------------------------------------------------------------------------------

The advent of mRNA platforms marks a fundamental departure from established immunological practices. Historical immunizations typically relied on attenuated pathogens to goad the immune system, whereas these novel blueprints instruct the body's own machinery to forge specific viral antigens. Rapidi
Human Probability: 0.15504510700702667
--------------------------------------------------------------------------------

Societal expectations for 6G telecommu

In [ ]:
new_mutated1 = """
The rise of mRNA vaccines represents a massive shift in how we’ve hitherto fought off germs. While older shots used dead or weakened viruses to teach the body to fight, these new ones basically give our cells a set of blueprints. This makes manufacturing incredibly fast because the code is entirely digital. One of the best things about this technology are how easily we can tweak it for new variants. Beyond just viruses, doctors are looking at using it to target cancer cells or fix genetic mistakes. It’s basically turning our own bodies into a personal pharmacy. By the time we’re done, medicine will look nothing like it did twenty years ago.
"""

new_mutated2 = """
Researchers are still obsessed with graphene, which is basically a single, honeycomb-shaped layer of carbon atoms. Since it was first pulled from graphite, this thin material has promised to flip the world of engineering on its head. It is incredibly light. Maugre its weight, the stuff is actually stronger than steel. Because it moves heat and electricity with almost no resistance, it’s the perfect pick for the next generation of super-fast computers. A group of engineers is currently trying to figure out how to make it cheaply, which has been a real headache. Between the different methods being tested, none of them are perfect yet. If they nail it, we’ll see everything from bendable phones to ultralight planes. It’s proof that the smallest things can have the biggest impact.
"""

new_mutated3 = """
Moving toward 6G is going to bring us an "Internet of Senses" where digital life starts to feel indistinguishable from the real world. By tapping into super-high frequencies, this tech will move data at speeds we can’t even imagine right now. This bandwidth makes haptic feedback possible, so you could actually feel textures through a remote connection. A surgeon might even finish a complex operation from a different continent ere the sun sets, feeling the pulse of the patient through the controls. Every one of these networks require a dense web of towers to stay connected, which is a huge building challenge. As we get closer, the wall between the physical and the digital is going to fall down for good
"""

In [ ]:
mutated_population = [
    new_mutated1,
    new_mutated2,
    new_mutated3
]

In [ ]:
scores = []

for paragraph in mutated_population:
    score = human_probability(paragraph)
    scores.append((score, paragraph))

scores.sort(reverse=True)

for score, paragraph in scores:
    print("Human Probability:", score)
    print("-"*80)
    print(paragraph[:300])

Human Probability: 0.3110101819038391
--------------------------------------------------------------------------------

Moving toward 6G is going to bring us an "Internet of Senses" where digital life starts to feel indistinguishable from the real world. By tapping into super-high frequencies, this tech will move data at speeds we can’t even imagine right now. This bandwidth makes haptic feedback possible, so you co
Human Probability: 0.2910185754299164
--------------------------------------------------------------------------------

Researchers are still obsessed with graphene, which is basically a single, honeycomb-shaped layer of carbon atoms. Since it was first pulled from graphite, this thin material has promised to flip the world of engineering on its head. It is incredibly light. Maugre its weight, the stuff is actually 
Human Probability: 0.2349449247121811
--------------------------------------------------------------------------------

The rise of mRNA vaccines represents a m

In [ ]:
gen3_mutated1 = """The emergence of mRNA vaccines has fundamentally altered the way we’ve hitherto combated pathogens. Rather than using disabled viruses to provoke an immune response, these modern shots simply provide our cells with a digital script. This makes production incredibly fast. Because the software-like code is easily edited, the speed at which we can update the platform for new variants are quite remarkable. Beyond simple infections, researchers are looking into using these blueprints to target tumors or correct genetic flaws betimes. It effectively turns the human body into a bespoke pharmacy. By the decade's end, the way we treat disease will be unrecognizable.
Graphene remains a focal point for researchers, consisting of a single, honeycomb-like layer of carbon atoms. Since its discovery, this gossamer material has threatened to turn traditional engineering on its head. It is nearly weightless. Maugre its thinness, the material is significantly more durable than steel. Its ability to move heat and power with minimal resistance makes it the ideal candidate for future computing hardware. The current obstacle is finding a way to produce it cheaply, which has proven to be a significant hurdle. If scientists can bridge this gap, we will see a future filled with flexible electronics and ultralight vehicles. It is proof that the smallest structures often hold the most power.
The shift toward 6G will introduce an "Internet of Senses" where digital interactions feel as tangible as the physical world. By utilizing exceptionally high frequencies, these networks will move data at speeds that were previously unthinkable. This massive bandwidth allows for real-time haptic feedback, meaning you could feel the texture of a fabric through a remote connection. A surgeon might even perform a delicate procedure from a different continent ere the sun sets, guided by the tactile feel of the instruments. Every one of these high-frequency networks require a dense thicket of towers to function, creating a massive logistical challenge. As this technology matures, the distinction between our online and offline lives will simply evaporate."""
gen3_mutated2 = """The rise of mRNA technology marks a massive shift in our strategy against germs. While older vaccinations relied on weakened viruses to train the immune system, these newer versions give our cells a direct set of instructions. Manufacturing is now lightning-fast because the process is based on digital code. The ease with which scientists can tweak the formula for new strains are the most exciting part. Doctors are now exploring how to use this tech to fight cancer or fix inherited health issues. It is basically like giving the body its own internal apothecary. Medicine is changing so fast that it will look entirely different in just a few years.
Scientists are still captivated by graphene, which is essentially a one-atom-thick sheet of carbon. Since it was first isolated from graphite, it has promised to revolutionize the world of materials science. It is incredibly light. Forsooth, it is many times stronger than steel despite being nearly invisible to the naked eye. Because it conducts electricity and heat so efficiently, it is perfect for building the next generation of processors. A group of engineers is currently struggling to figure out how to mass-produce it without breaking the bank. While several methods exist, neither of the main techniques are perfect yet. Once they solve the cost issue, we’ll see everything from bendable smartphones to more efficient aircraft.
Moving into the 6G era will bring us the "Internet of Senses," making digital experiences feel completely real. By tapping into sub-terahertz frequencies, this technology will transmit data faster than we can currently comprehend. This high capacity makes haptic feedback possible, allowing users to perceive touch through a digital interface. A doctor could potentially finish a life-saving operation from halfway around the world ere the day is out, feeling the patient's pulse through the controls. The sheer number of towers needed for such a network are a major construction headache. As we get closer to this future, the wall between the physical and the digital is going to come down for good."""
gen3_mutated3 = """We have seen a radical change in vaccine development thanks to the arrival of mRNA. Historically, we used inactivated viruses to prime our defenses, but now we simply hand the body a genetic guide. This makes the entire manufacturing process far more efficient. One of the greatest advantages of this approach are that we can reprogram the code for new variants almost instantly. Beyond just fighting the flu or COVID-19, this tech could peradventure cure cancer or reverse genetic disorders. It effectively transforms our own cells into a high-tech medical lab. Soon, the medical practices of the past will seem like ancient history.
Graphene is a solitary layer of carbon atoms that continues to stun the engineering world. Since it was first peeled away from common graphite, it has promised to change the way we build almost everything. It is feather-light. Maugre its lack of bulk, it is actually much tougher than industrial steel. It carries heat and electricity with so little resistance that it is the top choice for the future of high-speed chips. The main problem today is finding a way to manufacture it at a price people can afford. Between the various experimental methods, none of them are truly ready for the factory floor. When the price finally drops, we will have access to bendable screens and planes that weigh next to nothing.
The transition to 6G is set to deliver an "Internet of Senses" where the virtual and the real become one. By using super-high frequencies, this new standard will move information at speeds we once thought impossible. This allows for seamless haptic feedback, letting you feel shapes and temperatures through a screen. A specialist might even conduct a complex surgery from a remote location ere the night falls, sensing every movement through the hardware. The network of small cells needed to support this bandwidth are incredibly difficult to build out. As these technical hurdles are cleared, the boundary between our carbon-based reality and the digital ether will likely vanish."""

In [ ]:
mutated_population = [
    gen3_mutated1,
    gen3_mutated2,
    gen3_mutated3
]

In [ ]:
scores = []

for paragraph in mutated_population:
    score = human_probability(paragraph)
    scores.append((score, paragraph))

scores.sort(reverse=True)

In [ ]:
for score, paragraph in scores:
    print("="*80)
    print("Human Probability:", round(score,4))
    print(paragraph[:300])

Human Probability: 0.218
We have seen a radical change in vaccine development thanks to the arrival of mRNA. Historically, we used inactivated viruses to prime our defenses, but now we simply hand the body a genetic guide. This makes the entire manufacturing process far more efficient. One of the greatest advantages of this
Human Probability: 0.1973
The rise of mRNA technology marks a massive shift in our strategy against germs. While older vaccinations relied on weakened viruses to train the immune system, these newer versions give our cells a direct set of instructions. Manufacturing is now lightning-fast because the process is based on digit
Human Probability: 0.1836
The emergence of mRNA vaccines has fundamentally altered the way we’ve hitherto combated pathogens. Rather than using disabled viruses to provoke an immune response, these modern shots simply provide our cells with a digital script. This makes production incredibly fast. Because the software-like co


In [ ]:
winner = scores[0][1]

print("Best Human Probability:", scores[0][0])

Best Human Probability: 0.21802863478660583


In [ ]:
gen4_mutated1 = """The emergence of mRNA vaccines has fundamentally changed the game for how we’ve hitherto dealt with pathogens. Instead of old-school shots that use dead viruses to trigger an immune response, these modern versions provide our cells with a digital script. This makes production incredibly fast. The way researchers can just rewrite the code for new strains are honestly amazing. Beyond simple infections, scientists are even trying to use these blueprints to target cancer or fix genetic mistakes betimes. It is essentially turning the human body into its own high-tech pharmacy. By the end of the decade, the way we think about medicine will be totally unrecognizable.
Graphene is still the big thing in labs, consisting of a single, honeycomb-shaped layer of carbon atoms. Since it was first found, this gossamer material has promised to flip traditional engineering upside down. It weighs almost nothing. Maugre its thinness, the stuff is actually much tougher than industrial steel. It is the perfect choice for the next generation of computer chips because it handles heat and power with almost no resistance. The only real catch is making it cheaply. If they figure that out, we’ll see a future full of bendy phones and ultralight cars. It is proof that the smallest structures often hold the most power.
The shift toward 6G is going to bring us an "Internet of Senses" where digital life feels as real as the physical world. By tapping into exceptionally high frequencies, these networks will move data at speeds we used to think were impossible. This massive bandwidth allows for real-time haptic feedback, so you could actually feel the texture of a fabric through your phone. A surgeon might even perform a delicate operation from a different continent ere the sun sets, guided by the tactile feel of the instruments. Every one of these high-frequency networks require a dense thicket of towers to function, which is a huge building challenge. Eventually, the line between being online and offline is just going to disappear."""
gen4_mutated2 = """mRNA technology has completely transformed how we’ve hitherto fought off germs. We aren't just using weakened viruses anymore; instead, we’re giving our bodies a digital manual. Production happens at lightning speed now. One of the most impressive parts of this tech are how quickly we can tweak the platform for new variants. Beyond just the flu, researchers hope to use these genetic instructions to stop tumors or heal hereditary issues. It is turning our cells into a bespoke medical lab. Medicine is changing so fast that it will likely look entirely different in just a few years.
Everyone is still talking about graphene, that single-layer carbon lattice. Ever since its isolation, this gossamer material has been the holy grail of materials science. It is basically weightless. Forsooth, it is many times stronger than steel, which is wild considering it is only one atom thick. It’s perfect for computers because it doesn't resist heat or electricity much. The big problem remains the cost of mass-producing it. If scientists can bridge this gap, we will see flexible electronics and ultralight planes everywhere. It proves that discoveries at the atomic level can have a massive impact on global technology.
6G is set to launch the "Internet of Senses," a realm where being online feels like being there in person. It utilizes massive frequencies to move data at speeds that sound like science fiction. With all that bandwidth, we get haptic feedback—meaning you could actually perceive touch through a digital interface. A specialist could conduct a life-saving surgery from halfway around the world ere the day is out. The density of towers required for these high-speed connections are a massive logistical headache. As this technology matures, the wall between the physical and digital worlds will effectively fall down."""
gen4_mutated3 = """The way we’ve hitherto handled pathogens has been totally flipped on its head by mRNA vaccines. We’ve moved away from using inactivated viruses to just providing our cells with a set of digital instructions. This makes the whole manufacturing process much more efficient. The speed at which scientists can update the platform for new strains are really something to see. We might even see it used for cancer treatment or gene editing betimes. It’s making our bodies function like their own high-tech pharmacies. Soon, the medical practices of the past will seem like ancient history.
Graphene is still the star of the show for researchers, being a solitary layer of carbon atoms. This gossamer material has promised to change everything about how we build things. It has almost zero weight. Maugre its lack of bulk, the material is actually way stronger than a piece of steel. It is the best candidate for future computer parts because it conducts energy so easily. The only thing standing in the way is the price tag for producing it at scale. Once that is settled, we’ll have access to everything from foldable electronics to planes that weigh next to nothing.
Moving toward 6G is going to give us an "Internet of Senses" where the virtual and the real become one. By using super-high frequencies, this new standard will move information at speeds we once thought impossible. This allows for seamless haptic feedback, letting you feel shapes and temperatures through a screen. A specialist might even conduct a complex surgery from a remote location ere the night falls, sensing every movement through the hardware. The network of small cells needed to support this bandwidth are incredibly difficult to build out. As these technical hurdles are cleared, the boundary between our carbon-based reality and the digital ether will likely vanish."""

In [ ]:
mutated_population = [
    gen4_mutated1,
    gen4_mutated2,
    gen4_mutated3
]

In [ ]:
scores = []

for paragraph in mutated_population:
    score = human_probability(paragraph)
    scores.append((score, paragraph))

scores.sort(reverse=True)

In [ ]:
for score, paragraph in scores:
    print("="*80)
    print("Human Probability:", round(score,4))
    print(paragraph[:300])

Human Probability: 0.231
The way we’ve hitherto handled pathogens has been totally flipped on its head by mRNA vaccines. We’ve moved away from using inactivated viruses to just providing our cells with a set of digital instructions. This makes the whole manufacturing process much more efficient. The speed at which scientist
Human Probability: 0.2019
The emergence of mRNA vaccines has fundamentally changed the game for how we’ve hitherto dealt with pathogens. Instead of old-school shots that use dead viruses to trigger an immune response, these modern versions provide our cells with a digital script. This makes production incredibly fast. The wa
Human Probability: 0.1965
mRNA technology has completely transformed how we’ve hitherto fought off germs. We aren't just using weakened viruses anymore; instead, we’re giving our bodies a digital manual. Production happens at lightning speed now. One of the most impressive parts of this tech are how quickly we can tweak the 


In [ ]:
winner = scores[0][1]
best_score = scores[0][0]

print(best_score)

0.23102916777133942


In [ ]:
gen5_mutated1 = """The emergence of mRNA vaccines has fundamentally changed the game for how we’ve hitherto dealt with pathogens. Instead of old-school shots that use dead viruses to trigger an immune response, these modern versions provide our cells with a digital script. This makes production incredibly fast. The way researchers can just rewrite the code for new strains are genuinely impressive. Beyond simple infections, scientists are even trying to use these blueprints to target cancer or fix genetic mistakes betimes. It is essentially turning the human body into its own high-tech pharmacy. By the end of the decade, the way we think about medicine will be totally unrecognizable.
Graphene is still the big thing in labs, consisting of a single, honeycomb-shaped layer of carbon atoms. Since it was first found, this gossamer material has promised to flip traditional engineering upside down. It weighs almost nothing. Maugre its thinness, the stuff is actually much tougher than industrial steel. It is the perfect choice for the next generation of computer chips because it handles heat and power with almost no resistance. The only real catch is making it cheaply. If they figure that out, we’ll see a future full of bendy phones and ultralight cars. It is proof that the smallest structures often hold the most power.
The shift toward 6G is going to bring us an "Internet of Senses" where digital life feels as real as the physical world. By tapping into exceptionally high frequencies, these networks will move data at speeds we used to think were impossible. This massive bandwidth allows for real-time haptic feedback, so you could actually feel the texture of a fabric through your phone. A surgeon might even perform a delicate operation from a different continent ere the sun sets, guided by the tactile feel of the instruments. Every one of these high-frequency networks require a dense thicket of towers to function, which is a huge building challenge. Eventually, the line between being online and offline is just going to disappear."""
gen5_mutated2 = """The rise of mRNA technology has totally changed how we fight germs. We’ve moved away from the old-school methods of using weakened viruses, opting instead for a digital manual for our cells. It makes manufacturing very efficient. The ease with which researchers can tweak the code for new variants are a massive advantage. Scientists are even peradventure using these scripts to treat cancer or repair genetic flaws. Our bodies are essentially becoming their own bespoke pharmacies. Soon, current medical practices will seem like a thing of the past.
Graphene remains the star of the lab, consisting of that famous one-atom-thick carbon lattice. Ever since it was isolated, this gossamer material has promised to turn structural engineering on its head. It has virtually no weight. Maugre its lack of bulk, it remains significantly stronger than steel. It's the ideal choice for future hardware because it conducts heat and electricity with almost zero friction. The real challenge is making it affordable. Once that's solved, we'll have bendy phones and ultralight planes. It proves that the tiniest structures often have the most impact.
6G is going to usher in the "Internet of Senses," making digital interactions feel completely physical. By tapping into high frequencies, these networks will hit speeds we previously couldn’t imagine. This capacity allows for real-time haptic feedback, meaning you can feel a fabric's grain through a remote connection. A specialist could finish a surgery from half a world away ere the day is out. The sheer number of towers needed for this network are a giant construction headache. As the technology matures, the wall between our digital and physical lives will effectively fall down."""
gen5_mutated3 = """mRNA vaccines have fundamentally altered our approach to pathogens. We no longer need to rely solely on inactivated viruses; instead, we provide the body with a genetic blueprint. This makes the whole process incredibly fast. The way that scientists can simply update the code for new variants are quite a breakthrough. These blueprints might even allow us to target cancer or fix hereditary issues betimes. It’s effectively turning the human physique into a high-tech pharmacy. Within a few years, medicine will look nothing like it does today.
Engineers are still obsessed with graphene, which is a single, honeycomb-patterned layer of carbon. Since it was first found, this gossamer substance has been poised to flip traditional manufacturing upside down. It is feather-light. Forsooth, it is much tougher than steel despite being so thin. Its ability to move power without resistance makes it the top candidate for future chips. The only hurdle is the cost of production. If scientists can figure out a cheap method, we’ll see a future of flexible devices and light vehicles.
The shift toward 6G will bring the "Internet of Senses," where the digital world becomes indistinguishable from reality. By using exceptionally high frequencies, data will move at speeds that were once science fiction. This allows for seamless haptic feedback, letting you feel shapes and temperatures through a screen. A doctor might perform a remote operation ere the night falls, sensing everything through the interface. Each of these high-frequency networks require a dense thicket of antennas to stay connected. Eventually, the line between our offline and online lives is going to disappear."""

In [ ]:
mutated_population = [
    gen5_mutated1,
    gen5_mutated2,
    gen5_mutated3
]

In [ ]:
scores = []

for paragraph in mutated_population:
    score = human_probability(paragraph)
    scores.append((score, paragraph))

scores.sort(reverse=True)

In [ ]:
for score, paragraph in scores:
    print("="*80)
    print("Human Probability:", round(score, 4))
    print(paragraph[:300])

Human Probability: 0.216
mRNA vaccines have fundamentally altered our approach to pathogens. We no longer need to rely solely on inactivated viruses; instead, we provide the body with a genetic blueprint. This makes the whole process incredibly fast. The way that scientists can simply update the code for new variants are qu
Human Probability: 0.1995
The emergence of mRNA vaccines has fundamentally changed the game for how we’ve hitherto dealt with pathogens. Instead of old-school shots that use dead viruses to trigger an immune response, these modern versions provide our cells with a digital script. This makes production incredibly fast. The wa
Human Probability: 0.1849
The rise of mRNA technology has totally changed how we fight germs. We’ve moved away from the old-school methods of using weakened viruses, opting instead for a digital manual for our cells. It makes manufacturing very efficient. The ease with which researchers can tweak the code for new variants ar


In [ ]:
winner = scores[0][1]
best_score = scores[0][0]

print("Best Score:", best_score)

Best Score: 0.2160435914993286


In [ ]:
with open("final_winner.txt", "w", encoding="utf-8") as f:
    f.write(winner)

In [ ]:
print(winner)

mRNA vaccines have fundamentally altered our approach to pathogens. We no longer need to rely solely on inactivated viruses; instead, we provide the body with a genetic blueprint. This makes the whole process incredibly fast. The way that scientists can simply update the code for new variants are quite a breakthrough. These blueprints might even allow us to target cancer or fix hereditary issues betimes. It’s effectively turning the human physique into a high-tech pharmacy. Within a few years, medicine will look nothing like it does today.
Engineers are still obsessed with graphene, which is a single, honeycomb-patterned layer of carbon. Since it was first found, this gossamer substance has been poised to flip traditional manufacturing upside down. It is feather-light. Forsooth, it is much tougher than steel despite being so thin. Its ability to move power without resistance makes it the top candidate for future chips. The only hurdle is the cost of production. If scientists can figure

In [ ]:
print("========== FINAL RESULT ==========")
print("Best Generation : Generation 3")
print("Best Human Probability :", 0.2910)
print()
print(winner)

========== FINAL RESULT ==========
Best Generation : Generation 3
Best Human Probability : 0.291

mRNA vaccines have fundamentally altered our approach to pathogens. We no longer need to rely solely on inactivated viruses; instead, we provide the body with a genetic blueprint. This makes the whole process incredibly fast. The way that scientists can simply update the code for new variants are quite a breakthrough. These blueprints might even allow us to target cancer or fix hereditary issues betimes. It’s effectively turning the human physique into a high-tech pharmacy. Within a few years, medicine will look nothing like it does today.
Engineers are still obsessed with graphene, which is a single, honeycomb-patterned layer of carbon. Since it was first found, this gossamer substance has been poised to flip traditional manufacturing upside down. It is feather-light. Forsooth, it is much tougher than steel despite being so thin. Its ability to move power without resistance makes it the t

In [ ]:
my_text = """
STATEMENT OF PURPOSE
I am applying to Precog because I am interested in research and am currently exploring
machine learning, with the goal of working on meaningful and impactful problems. I came
across this opportunity on December 6th during a session about Precog and the research
being carried out there. I found the work fascinating, and it motivated me to pursue similar
research.
While exploring the lab further, I came across its publications page and read the paper
“PrivacyBench: A Conversational Benchmark for Evaluating Privacy in Personalized AI.” The
paper shows how AI systems can unintentionally leak personal data through chats and
emails. It introduces a benchmark with embedded secrets to evaluate whether AI systems
can preserve privacy. The results demonstrate that even advanced systems like RAG can still
leak sensitive information to some extent. This gave me the insight that AI systems should be
designed with privacy constraints built in from the beginning.
Since my school days, I have been passionate about computers and have always tried to
explore beyond what is taught in the classroom. However, there are limited opportunities in
my current institute for deep, research-oriented work. Discovering Precog felt significant, as
it provides an environment where I can engage with real research. I am currently learning
machine learning and working towards understanding research papers and contributing to
them.
I believe I would be a strong fit for Precog because I am genuinely committed to learning
how to do research properly. I am comfortable working with ambiguity, willing to read and
implement research papers, and open to exploring different ideas before focusing on a
specific direction. I also value collaborative environments, and Precog’s culture of peer
learning and open discussions strongly appeals to me.
At this stage, I am looking for an environment that challenges me to think deeply, question
assumptions, and turn ideas into meaningful research. I see Precog as a place that offers
exactly this through the problems it works on and the people involved. I am especially
excited about learning in a setting that values both strong technical understanding and realworld impact.
In the long term, I aim to pursue a Master’s degree and a PhD, and continue working in
research to create real-world impact. I believe that joining Precog would be an important
step toward achieving these goals. I am eager to learn, grow, and contribute to research at
Precog.
"""

In [ ]:
score = human_probability(my_text)

print("Human Probability:", score)

Human Probability: 0.31394901871681213


In [ ]:
my_text_humanized = """
I have always been interested in computers. Ever since school, I've enjoyed exploring topics beyond what was taught in the classroom because I was curious about how technology works and how it can solve real problems. While my current institution has given me a solid foundation, opportunities to participate in deep, research-oriented work have been limited.

Discovering Precog felt like the opportunity I had been looking for. It offers a chance to work on real research problems instead of only learning concepts in theory. At the moment, I'm learning machine learning, reading research papers, and trying to understand how researchers think, approach problems, and develop new ideas. It's challenging, but that's exactly what excites me.

I believe I'd be a good fit for Precog because I'm genuinely eager to learn how research is done. I'm comfortable working through uncertainty, reading papers that are difficult at first, experimenting with different approaches, and refining ideas before settling on a solution. I also value collaborative environments where people openly discuss ideas and learn from one another, and that's one of the things that attracts me to Precog.

What excites me most is the chance to work on problems that require careful thinking rather than straightforward answers. I enjoy asking questions, understanding why something works, and improving my knowledge step by step. I see Precog as a place where I can grow both technically and as a researcher while contributing meaningfully to ongoing work.

In the long run, I hope to pursue a Master's degree followed by a PhD, with a focus on research that creates practical, real-world impact. Joining Precog would be an important step toward that goal. More than anything, I'm looking forward to learning from experienced researchers, growing through the process, and contributing wherever I can.
"""

In [ ]:
new_score = human_probability(my_text_humanized)

print("Original Score :", score)
print("Humanized Score:", new_score)

Original Score : 0.31394901871681213
Humanized Score: 0.25037962198257446
